**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 22 - SQL y manejo de tablas**

## Complemento

Este notebook se complementa con la presentación: **DATA.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

## ¿Qué vamos a hacer en esta clase?

Todo lo que vimos hasta ahora arrancaba con un archivo. Pero en una organización de verdad los datos
**no viven en archivos sueltos**: viven en una **base de datos**. Y a las bases de datos se les habla
en un idioma que tiene 50 años y no piensa jubilarse: **SQL**.

| Parte | Tema | Cláusula |
|---|---|---|
| **A** | Por qué SQL y no un Excel | — |
| **B** | Armar la base y mirarla | `sqlite3` |
| **C** | Traer datos y filtrar | `SELECT` · `WHERE` · `ORDER BY` |
| **D** | Resumir | `GROUP BY` · `HAVING` |
| **E** | **Cruzar tablas** | `JOIN` |
| **F** | Preguntas dentro de preguntas | Subconsultas |
| **G** | El diccionario SQL ↔ pandas | — |

> **Por qué te sirve:** "¿sabés SQL?" es literalmente la primera pregunta en una entrevista de análisis
> de datos, control de gestión o auditoría. Es la habilidad técnica más pedida y la más fácil de aprender
> de todas las que vimos.

> ✅ **No hay que instalar nada.** Usamos `sqlite3`, que ya viene adentro de Python.

In [ ]:
import sqlite3                     # sqlite3: motor de base de datos incluido en Python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
URL = "https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/main/DF/"

---
# 🤔 Parte A — ¿Por qué SQL si ya sé pandas?

Las dos herramientas hacen lo mismo. La diferencia está en **dónde viven los datos**.

| | pandas | SQL |
|---|---|---|
| Dónde están los datos | En la memoria de tu computadora | En un servidor, ordenados |
| Cuánto aguanta | Lo que entre en tu RAM | Millones de filas sin despeinarse |
| Quién más lo usa | Analistas | Analistas, sistemas, la empresa entera |
| Varias personas a la vez | No | Sí |

**En criollo:** pandas es tu escritorio, SQL es el archivo central de la empresa. Vas al archivo central,
pedís *solo lo que necesitás*, y eso lo traés a tu escritorio para trabajarlo.

### Un poco de vocabulario

| Término | En Excel sería | En pandas sería |
|---|---|---|
| **Base de datos** | El archivo entero | — |
| **Tabla** | Una hoja | Un DataFrame |
| **Registro** (fila) | Una fila | Una fila |
| **Campo** (columna) | Una columna | Una columna |
| **Clave primaria** | La columna de ID única | El índice |
| **Consulta** (*query*) | Un filtro o tabla dinámica | Una línea de código |

---
# 🗄️ Parte B — Armamos nuestra base de datos

Vamos a construir la base de una distribuidora de tecnología con **tres tablas**:

| Tabla | Qué guarda | Filas |
|---|---|---|
| `ventas` | Cada operación de venta | 200 |
| `productos` | El catálogo: categoría y costo | 6 |
| `vendedores` | El equipo comercial y su sucursal | 5 |

Esta forma de organizar —una tabla de **hechos** rodeada de tablas de **referencia**— es la más común
en cualquier empresa. Se llama *modelo estrella*.

In [ ]:
# 1) Traemos las ventas del repositorio
ventas = pd.read_csv(URL + "ventas.csv")
ventas["Total"] = ventas["Cantidad"] * ventas["Precio_Unitario"]

# 2) Creamos el catálogo de productos
productos = pd.DataFrame({
    "Producto":  ["Notebook", "Monitor", "Impresora", "Teclado", "Mouse", "Auriculares"],
    "Categoria": ["Cómputo", "Cómputo", "Periférico", "Periférico", "Periférico", "Audio"],
    "Costo":     [62000, 31000, 24000, 8500, 4200, 6800],
})

# 3) Y el equipo de ventas
vendedores = pd.DataFrame({
    "Vendedor":  ["Juan", "Lucía", "María", "Pedro", "Sofía"],
    "Sucursal":  ["Centro", "Norte", "Centro", "Sur", "Norte"],
    "Antiguedad": [8, 3, 12, 1, 5],
})

print("ventas:", ventas.shape, "| productos:", productos.shape, "| vendedores:", vendedores.shape)
ventas.head(3)

In [ ]:
# Creamos la base de datos en memoria y volcamos las tres tablas
con = sqlite3.connect(":memory:")     # ":memory:" = base temporal en RAM (se borra al cerrar)

ventas.to_sql("ventas", con, index=False, if_exists="replace")
productos.to_sql("productos", con, index=False, if_exists="replace")
vendedores.to_sql("vendedores", con, index=False, if_exists="replace")

print("Base creada ✅")

> 📌 **`:memory:`** crea una base que vive solo mientras dure la sesión. Si querés un archivo de verdad
> en el disco, se pone el nombre: `sqlite3.connect("empresa.db")`. Todo lo demás es igual.

In [ ]:
# Esta función la vamos a usar en toda la clase: manda una consulta y devuelve un DataFrame
def sql(consulta):
    return pd.read_sql_query(consulta, con)

# Probémosla: ¿qué tablas tiene la base?
sql("SELECT name FROM sqlite_master WHERE type='table'")

---
# 🔍 Parte C — `SELECT`: traer datos

La estructura básica de toda consulta SQL es siempre la misma:

```sql
SELECT   columnas          -- qué quiero
FROM     tabla             -- de dónde
WHERE    condición         -- qué filas
ORDER BY columna           -- cómo ordenado
LIMIT    n                 -- cuántas
```

Se lee casi como una oración en inglés. Empecemos por lo mínimo.

In [ ]:
# El asterisco * significa "todas las columnas"
sql("SELECT * FROM ventas LIMIT 5")

In [ ]:
# Pedimos solo algunas columnas, en el orden que queramos
sql("""
    SELECT Fecha, Producto, Vendedor, Total
    FROM ventas
    LIMIT 5
""")

> 💡 Las **comillas triples** `""" """` permiten escribir la consulta en varias líneas.
> A partir de acá las usamos siempre: una consulta SQL bien indentada se lee muchísimo mejor.
>
> Por convención, las **palabras de SQL van en mayúscula** (`SELECT`, `FROM`, `WHERE`) y los nombres de
> tablas y columnas en minúscula. No es obligatorio, pero todo el mundo lo hace.

### `WHERE`: filtrar filas

Es el equivalente exacto de `ventas[ventas["Ciudad"] == "Córdoba"]` en pandas.

In [ ]:
sql("""
    SELECT Fecha, Producto, Vendedor, Total
    FROM ventas
    WHERE Ciudad = 'Córdoba'
    LIMIT 5
""")

| Operador | Qué hace | Ejemplo |
|---|---|---|
| `=` | Igual (¡uno solo, no `==`!) | `WHERE Ciudad = 'Rosario'` |
| `<>` o `!=` | Distinto | `WHERE Producto <> 'Mouse'` |
| `>` `<` `>=` `<=` | Comparación | `WHERE Total > 100000` |
| `AND` / `OR` | Combinar condiciones | `WHERE Total > 100000 AND Ciudad = 'Rosario'` |
| `IN` | Está en una lista | `WHERE Producto IN ('Monitor', 'Notebook')` |
| `BETWEEN` | Entre dos valores | `WHERE Cantidad BETWEEN 3 AND 6` |
| `LIKE` | Coincide con un patrón | `WHERE Producto LIKE 'Note%'` |
| `IS NULL` | Falta el dato | `WHERE Ciudad IS NULL` |

> ⚠️ **Dos trampas clásicas para el que viene de Python:**
> 1. En SQL la igualdad es **`=`**, no `==`.
> 2. El texto va entre **comillas simples**: `'Córdoba'`.

In [ ]:
# Varias condiciones a la vez
sql("""
    SELECT Fecha, Producto, Ciudad, Cantidad, Total
    FROM ventas
    WHERE Total > 300000
      AND Producto IN ('Notebook', 'Monitor')
    ORDER BY Total DESC
    LIMIT 8
""")

`ORDER BY Total DESC` ordena de mayor a menor (`DESC` = *descending*). Sin el `DESC` ordena de menor a mayor.

---
# 📊 Parte D — `GROUP BY`: resumir

Acá SQL empieza a brillar. Las **funciones de agregación** toman muchas filas y devuelven un solo número:

| Función | Qué calcula |
|---|---|
| `COUNT(*)` | Cuántas filas |
| `SUM(col)` | La suma |
| `AVG(col)` | El promedio |
| `MIN(col)` / `MAX(col)` | El mínimo / máximo |
| `ROUND(col, 2)` | Redondear |

In [ ]:
# Sin GROUP BY: un resumen de TODA la tabla
sql("""
    SELECT COUNT(*)          AS operaciones,
           SUM(Total)        AS facturacion,
           ROUND(AVG(Total)) AS ticket_promedio,
           MAX(Total)        AS venta_mas_grande
    FROM ventas
""")

`AS` le pone nombre a la columna del resultado. Sin eso, la columna se llamaría `COUNT(*)`, que es feo
e incómodo de usar después.

In [ ]:
# Con GROUP BY: el mismo resumen, pero abierto por vendedor
sql("""
    SELECT Vendedor,
           COUNT(*)          AS operaciones,
           SUM(Total)        AS facturacion,
           ROUND(AVG(Total)) AS ticket_promedio
    FROM ventas
    GROUP BY Vendedor
    ORDER BY facturacion DESC
""")

**La regla de oro del `GROUP BY`:** toda columna que aparezca en el `SELECT` y **no** esté dentro de una
función de agregación, tiene que estar en el `GROUP BY`. Es el error más común al empezar.

In [ ]:
# Se puede agrupar por más de una columna
sql("""
    SELECT Ciudad,
           Producto,
           COUNT(*)   AS operaciones,
           SUM(Total) AS facturacion
    FROM ventas
    GROUP BY Ciudad, Producto
    ORDER BY facturacion DESC
    LIMIT 10
""")

### `HAVING`: filtrar **después** de agrupar

Esta distinción entra en el parcial, así que prestale atención:

| Cláusula | Cuándo actúa | Filtra |
|---|---|---|
| `WHERE` | **Antes** de agrupar | Filas individuales |
| `HAVING` | **Después** de agrupar | Grupos ya resumidos |

*"Ciudades cuya facturación total supere el millón"* → eso solo se sabe **después** de sumar. Va en `HAVING`.

In [ ]:
sql("""
    SELECT Ciudad,
           COUNT(*)   AS operaciones,
           SUM(Total) AS facturacion
    FROM ventas
    WHERE Cantidad >= 3            -- filtra FILAS antes de agrupar
    GROUP BY Ciudad
    HAVING SUM(Total) > 3000000    -- filtra GRUPOS después de agrupar
    ORDER BY facturacion DESC
""")

---
# 🔗 Parte E — `JOIN`: cruzar tablas

Este es **el corazón de SQL** y la razón por la que las bases de datos se dividen en varias tablas.

La tabla `ventas` sabe que se vendió una "Notebook", pero **no sabe cuánto cuesta producirla**.
Eso está en `productos`. Para calcular la ganancia hay que **pegar** las dos tablas por la columna
que tienen en común: `Producto`.

```
    ventas                    productos
┌───────────┬───────┐     ┌───────────┬──────────┬───────┐
│ Producto  │ Total │     │ Producto  │Categoria │ Costo │
├───────────┼───────┤     ├───────────┼──────────┼───────┤
│ Notebook  │ 614k  │ ←→  │ Notebook  │ Cómputo  │ 62000 │
│ Monitor   │ 138k  │ ←→  │ Monitor   │ Cómputo  │ 31000 │
└───────────┴───────┘     └───────────┴──────────┴───────┘
                  ↑ la columna en común
```

In [ ]:
sql("""
    SELECT v.Fecha,
           v.Producto,
           p.Categoria,
           v.Cantidad,
           v.Total,
           p.Costo * v.Cantidad         AS costo_total,
           v.Total - p.Costo * v.Cantidad AS ganancia
    FROM ventas v
    JOIN productos p  ON v.Producto = p.Producto
    LIMIT 8
""")

**Desarmemos la consulta:**

- `FROM ventas v` → la tabla `ventas`, a la que le ponemos el apodo (*alias*) `v`.
- `JOIN productos p` → le pegamos `productos`, con el alias `p`.
- `ON v.Producto = p.Producto` → **la condición del cruce**: unir las filas donde el producto coincida.
- `v.Total`, `p.Costo` → el alias evita ambigüedad cuando las dos tablas tienen columnas con el mismo nombre.

> ⚠️ **Si te olvidás el `ON`**, SQL cruza *cada fila con cada fila*: 200 × 6 = 1.200 filas de basura.
> Se llama *producto cartesiano* y es el error más caro de SQL.

In [ ]:
# Ahora la pregunta de negocio: ¿qué categoría deja más margen?
sql("""
    SELECT p.Categoria,
           COUNT(*)                                AS operaciones,
           SUM(v.Total)                            AS facturacion,
           SUM(v.Total - p.Costo * v.Cantidad)     AS ganancia,
           ROUND(100.0 * SUM(v.Total - p.Costo * v.Cantidad) / SUM(v.Total), 1) AS margen_pct
    FROM ventas v
    JOIN productos p ON v.Producto = p.Producto
    GROUP BY p.Categoria
    ORDER BY ganancia DESC
""")

> 💡 Fijate el `100.0` en lugar de `100`. En SQLite, dividir dos enteros da un entero
> (`3 / 2 = 1`). Poniendo un decimal forzás la división real. Es una trampa clásica.

In [ ]:
# Se pueden encadenar varios JOIN: ventas + productos + vendedores
sql("""
    SELECT ve.Sucursal,
           p.Categoria,
           COUNT(*)     AS operaciones,
           SUM(v.Total) AS facturacion
    FROM ventas v
    JOIN productos  p  ON v.Producto = p.Producto
    JOIN vendedores ve ON v.Vendedor = ve.Vendedor
    GROUP BY ve.Sucursal, p.Categoria
    ORDER BY ve.Sucursal, facturacion DESC
""")

### Los tipos de `JOIN`

| Tipo | Qué devuelve | Cuándo usarlo |
|---|---|---|
| `INNER JOIN` (o solo `JOIN`) | Solo lo que coincide en **ambas** tablas | Por defecto |
| `LEFT JOIN` | **Todo** lo de la izquierda + lo que matchee de la derecha | Para no perder filas |
| `RIGHT` / `FULL` | Al revés / todo | SQLite no los soporta; otros motores sí |

**Cuándo importa:** si un producto de `ventas` no estuviera en el catálogo, el `INNER JOIN`
lo **borraría en silencio**. El `LEFT JOIN` lo conserva con `NULL` en las columnas del catálogo.

### 🔵 Los `JOIN`, en un dibujo

Cada tipo de `JOIN` se distingue por **qué filas conserva** cuando una tabla tiene algo que la otra
no. Pensá en dos conjuntos: la tabla **A** (la de la izquierda) y la tabla **B** (la de la derecha).
Lo sombreado es lo que queda en el resultado.

<svg viewBox="0 0 620 140" width="100%" style="max-width:620px">

  <g transform="translate(0,0)">
    <text x="60" y="16" text-anchor="middle" font-size="12" font-weight="bold" fill="#334155">INNER JOIN</text>
    <circle cx="45" cy="70" r="36" fill="none" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <circle cx="75" cy="70" r="36" fill="none" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <path d="M 60 40 A 36 36 0 0 1 60 100 A 36 36 0 0 1 60 40"
          fill="#1d4ed8" fill-opacity="0.55" stroke="none"/>
    <text x="30" y="120" font-size="11" fill="#64748b">A</text>
    <text x="88" y="120" font-size="11" fill="#64748b">B</text>
  </g>
  <g transform="translate(125,0)">
    <text x="60" y="16" text-anchor="middle" font-size="12" font-weight="bold" fill="#334155">LEFT JOIN</text>
    <circle cx="45" cy="70" r="36" fill="#2563eb" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <circle cx="75" cy="70" r="36" fill="none" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <path d="M 60 40 A 36 36 0 0 1 60 100 A 36 36 0 0 1 60 40"
          fill="#1d4ed8" fill-opacity="0.55" stroke="none"/>
    <text x="30" y="120" font-size="11" fill="#64748b">A</text>
    <text x="88" y="120" font-size="11" fill="#64748b">B</text>
  </g>
  <g transform="translate(250,0)">
    <text x="60" y="16" text-anchor="middle" font-size="12" font-weight="bold" fill="#334155">RIGHT JOIN</text>
    <circle cx="45" cy="70" r="36" fill="none" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <circle cx="75" cy="70" r="36" fill="#2563eb" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <path d="M 60 40 A 36 36 0 0 1 60 100 A 36 36 0 0 1 60 40"
          fill="#1d4ed8" fill-opacity="0.55" stroke="none"/>
    <text x="30" y="120" font-size="11" fill="#64748b">A</text>
    <text x="88" y="120" font-size="11" fill="#64748b">B</text>
  </g>
  <g transform="translate(375,0)">
    <text x="60" y="16" text-anchor="middle" font-size="12" font-weight="bold" fill="#334155">FULL JOIN</text>
    <circle cx="45" cy="70" r="36" fill="#2563eb" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <circle cx="75" cy="70" r="36" fill="#2563eb" fill-opacity="0.35" stroke="#2563eb" stroke-width="2"/>
    <path d="M 60 40 A 36 36 0 0 1 60 100 A 36 36 0 0 1 60 40"
          fill="#1d4ed8" fill-opacity="0.55" stroke="none"/>
    <text x="30" y="120" font-size="11" fill="#64748b">A</text>
    <text x="88" y="120" font-size="11" fill="#64748b">B</text>
  </g>
  <g transform="translate(500,0)">
    <text x="60" y="16" text-anchor="middle" font-size="12" font-weight="bold" fill="#334155">CROSS JOIN</text>
    <g stroke="#7c3aed" stroke-width="2" fill="#f5f3ff">
      <rect x="18" y="40" width="34" height="22"/><rect x="18" y="66" width="34" height="22"/>
      <rect x="70" y="40" width="34" height="22"/><rect x="70" y="66" width="34" height="22"/>
    </g>
    <line x1="52" y1="51" x2="70" y2="51" stroke="#7c3aed" stroke-width="1.5"/>
    <line x1="52" y1="51" x2="70" y2="77" stroke="#7c3aed" stroke-width="1.5"/>
    <line x1="52" y1="77" x2="70" y2="51" stroke="#7c3aed" stroke-width="1.5"/>
    <line x1="52" y1="77" x2="70" y2="77" stroke="#7c3aed" stroke-width="1.5"/>
    <text x="60" y="120" text-anchor="middle" font-size="11" fill="#64748b">todas × todas</text>
  </g>
</svg>

| Tipo | Qué devuelve | Cuándo se usa en la práctica |
|---|---|---|
| **`INNER JOIN`** | Solo lo que está **en las dos** tablas | El más común. Ventas que tienen cliente identificado |
| **`LEFT JOIN`** | **Todo A**, y de B lo que matchee (si no, `NULL`) | *"Todos los clientes, hayan comprado o no"* |
| **`RIGHT JOIN`** | **Todo B**, y de A lo que matchee | Casi no se usa: se da vuelta el orden y se usa `LEFT` |
| **`FULL JOIN`** | **Todo de las dos**, matcheado donde se pueda | Conciliar dos sistemas y ver qué falta en cada uno |
| **`CROSS JOIN`** | **Todas las combinaciones** posibles | Armar una grilla completa: todos los meses × todas las sucursales |

> 🚨 **El error más caro de SQL vive acá.** Si usás `INNER JOIN` para cruzar ventas con clientes y
> hay ventas sin cliente cargado, **esas ventas desaparecen del informe** y nadie se entera: el total
> simplemente da menos. Cuando el número no cierra por poco, **empezá sospechando del `JOIN`**.
>
> 💡 **El control:** contá las filas antes y después del cruce. Si bajaron, perdiste datos; si
> subieron, la clave estaba duplicada y **estás contando ventas de más**.

In [ ]:
# Comprobamos que no estemos perdiendo ventas en el cruce
sql("""
    SELECT (SELECT COUNT(*) FROM ventas)                                    AS filas_originales,
           (SELECT COUNT(*) FROM ventas v JOIN productos p
                            ON v.Producto = p.Producto)                     AS filas_tras_join
""")

Dan igual: no perdimos ninguna venta. **Hacé siempre este control después de un `JOIN`.**

---
# 🪆 Parte F — Subconsultas

Una consulta puede ir **adentro** de otra. Sirve para preguntas de dos pasos, del tipo:
*"¿qué ventas superan el promedio general?"* — primero hay que calcular el promedio.

In [ ]:
sql("""
    SELECT Fecha, Producto, Vendedor, Total
    FROM ventas
    WHERE Total > (SELECT AVG(Total) FROM ventas)    -- la subconsulta calcula el promedio
    ORDER BY Total DESC
    LIMIT 8
""")

In [ ]:
# Una subconsulta puede hacer de tabla temporal
sql("""
    SELECT Sucursal,
           ROUND(AVG(facturacion)) AS facturacion_promedio_por_vendedor
    FROM (
        SELECT ve.Sucursal, v.Vendedor, SUM(v.Total) AS facturacion
        FROM ventas v
        JOIN vendedores ve ON v.Vendedor = ve.Vendedor
        GROUP BY ve.Sucursal, v.Vendedor
    )
    GROUP BY Sucursal
    ORDER BY facturacion_promedio_por_vendedor DESC
""")

In [ ]:
# El resultado de sql() es un DataFrame de pandas → se grafica como cualquier otro
datos = sql("""
    SELECT Ciudad, SUM(Total) AS facturacion
    FROM ventas
    GROUP BY Ciudad
    ORDER BY facturacion DESC
""")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(datos["Ciudad"], datos["facturacion"], color="#243b5e")
ax.set_title("Facturación por ciudad", loc="left", fontweight="bold")
ax.set_ylabel("Facturación ($)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

---
# 🔄 Parte G — El diccionario SQL ↔ pandas

Las dos herramientas hacen lo mismo con distinta sintaxis. Guardate esta tabla:

| Objetivo | SQL | pandas |
|---|---|---|
| Ver todo | `SELECT * FROM t` | `df` |
| Algunas columnas | `SELECT a, b FROM t` | `df[["a", "b"]]` |
| Filtrar | `WHERE a > 5` | `df[df["a"] > 5]` |
| Ordenar | `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| Primeras n | `LIMIT 5` | `df.head(5)` |
| Contar | `COUNT(*)` | `len(df)` |
| Agrupar y sumar | `GROUP BY a` + `SUM(b)` | `df.groupby("a")["b"].sum()` |
| Filtrar grupos | `HAVING SUM(b) > 100` | `.groupby("a").filter(...)` |
| Cruzar tablas | `JOIN t2 ON t.k = t2.k` | `df.merge(df2, on="k")` |
| Valores únicos | `SELECT DISTINCT a` | `df["a"].unique()` |

> 🎯 **En la práctica se usan juntas:** SQL para traer del servidor solo lo que necesitás
> (rápido, poca memoria), y pandas para analizar y graficar. Es exactamente lo que hicimos hoy.

---
## 📜 Qué es un *script* y qué es un *esquema*

Hasta acá escribimos consultas sueltas. Pero en una organización, SQL **no se usa así**: se usa en
**scripts**.

### El script

Un **script** es simplemente **un archivo de texto con varias instrucciones SQL, en orden**, que se
ejecutan una atrás de la otra. La misma idea que un notebook, pero en un `.sql`.

**¿Por qué importa?** Porque una base de datos no se arma a mano clickeando: se arma **corriendo un
script**. Eso tiene tres consecuencias enormes:

| Sin script | Con script |
|---|---|
| Alguien creó las tablas a mano y nadie sabe cómo | La estructura está **escrita y versionada** |
| Rearmar el ambiente de prueba lleva días | Se corre el script y en un minuto está |
| "En mi máquina anda" | Todos parten **exactamente** de lo mismo |

> 🎯 **Es el mismo principio que venimos usando toda la materia:** si está escrito en código, es
> reproducible; si se hizo a mano, se perdió.

In [ ]:
# Un script típico: crea la estructura, la llena y consulta. Todo junto y en orden.
script = """
-- 1) Definimos la estructura
CREATE TABLE IF NOT EXISTS sucursales (
    id      INTEGER PRIMARY KEY,
    nombre  TEXT    NOT NULL,
    region  TEXT
);

CREATE TABLE IF NOT EXISTS ventas (
    id           INTEGER PRIMARY KEY,
    sucursal_id  INTEGER REFERENCES sucursales(id),   -- clave foránea
    importe      REAL    NOT NULL,
    fecha        TEXT
);

-- 2) Cargamos datos
INSERT INTO sucursales (id, nombre, region) VALUES
    (1, 'Centro',   'AMBA'),
    (2, 'Norte',    'AMBA'),
    (3, 'Córdoba',  'Interior');

INSERT INTO ventas (id, sucursal_id, importe, fecha) VALUES
    (1, 1, 145000, '2026-03-01'),
    (2, 1,  98000, '2026-03-04'),
    (3, 2, 210000, '2026-03-02'),
    (4, 3,  76000, '2026-03-05');
"""

import sqlite3
import pandas as pd

conexion = sqlite3.connect(":memory:")     # una base que vive en memoria, para practicar
conexion.executescript(script)             # ← executescript corre TODO el script de una

print("Base creada. Tablas que existen ahora:")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conexion))

### El esquema

El **esquema** (*schema*) es **el plano de la base**: qué tablas hay, qué columnas tiene cada una, de
qué tipo son y **cómo se relacionan entre sí**.

En el script de arriba, el esquema son los `CREATE TABLE`. Fijate tres cosas que aparecen ahí y que
son las que le dan solidez a una base:

| Elemento | Qué hace | Por qué importa |
|---|---|---|
| **`PRIMARY KEY`** | Identifica cada fila de forma única | Sin esto, terminás con la misma venta cargada dos veces |
| **`REFERENCES`** (clave foránea) | Conecta una tabla con otra | Es **lo que hace posible el `JOIN`** |
| **`NOT NULL`** | Prohíbe que el campo quede vacío | La base **rechaza** el dato malo en vez de guardarlo |

> 💼 **Acá está la diferencia grande con Excel.** En una planilla, cualquiera escribe cualquier cosa
> en cualquier celda. En una base con un buen esquema, **el dato inconsistente ni siquiera entra**:
> la base lo rechaza. Por eso los sistemas serios de una organización corren sobre bases de datos y
> no sobre planillas.

In [ ]:
# El esquema se puede consultar: la base sabe describirse a sí misma
print(pd.read_sql("PRAGMA table_info(ventas)", conexion))

print("\nY así se ve el JOIN que las claves hacen posible:")
consulta = """
SELECT  s.nombre           AS sucursal,
        s.region,
        COUNT(v.id)        AS operaciones,
        SUM(v.importe)     AS facturado
FROM        sucursales s
LEFT JOIN   ventas     v ON v.sucursal_id = s.id
GROUP BY    s.nombre, s.region
ORDER BY    facturado DESC
"""
print(pd.read_sql(consulta, conexion))

> 🔍 **Mirá el resultado de arriba con atención.** Usamos `LEFT JOIN`, así que aparecen **las tres
> sucursales**, incluso las que tienen pocas ventas. Si hubiéramos usado `INNER JOIN` y alguna
> sucursal no tuviera ninguna venta cargada, **habría desaparecido del informe** — y el gerente de
> esa sucursal se enteraría de que "no existe" en la reunión.

---
# 📝 Ejercicios

Trabajá con la base que ya está cargada. Usá la función `sql("...")`.

**Ejercicio 1.** Traé las 10 ventas de mayor monto, mostrando fecha, producto, vendedor y total.

In [ ]:
# Tu respuesta acá

**Ejercicio 2.** ¿Cuántas operaciones y cuánto facturó cada **ciudad**? Ordenado de mayor a menor.

In [ ]:
# Tu respuesta acá

**Ejercicio 3.** ¿Qué productos tuvieron un ticket promedio mayor a \$150.000?
Cuidado: ¿va en `WHERE` o en `HAVING`?

In [ ]:
# Tu respuesta acá

**Ejercicio 4.** Con un `JOIN` a `vendedores`, calculá la facturación por **sucursal** y la
antigüedad promedio de su equipo.

In [ ]:
# Tu respuesta acá

**Ejercicio 5.** ¿Cuál es el producto **más rentable** en pesos? (necesitás cruzar con `productos`
y restar el costo).

In [ ]:
# Tu respuesta acá

**Ejercicio 6 (integrador).** Resolvé el Ejercicio 2 **otra vez**, pero con pandas en lugar de SQL.
Compará los dos resultados con `.equals()` y decidí cuál te resultó más cómodo escribir.

In [ ]:
# Tu respuesta acá

**Ejercicio 7 (desafío 🥇⚡🤓).** Para cada vendedor, calculá qué porcentaje de **su** facturación
viene de la categoría "Cómputo". Pista: vas a necesitar una subconsulta o un `JOIN` con un `GROUP BY` adentro.

In [ ]:
# Tu respuesta acá

In [ ]:
con.close()   # buena práctica: cerrar la conexión cuando terminás
print("Conexión cerrada ✅")

---
## 🏢 Cómo se organiza una empresa de software (y por qué te importa)

Cuando trabajes con datos en una organización, **la base de datos no te la vas a encontrar sola**:
es una pieza de un sistema más grande, hecho y mantenido por un equipo. Estas palabras se van a usar
delante tuyo el primer día, así que conviene tenerlas claras.

### Las dos mitades de cualquier sistema

| | **Front-end** | **Back-end** |
|---|---|---|
| **Qué es** | Lo que el usuario ve y toca | La maquinaria que no se ve |
| **Dónde corre** | En la pantalla del usuario | En un servidor |
| **De qué se ocupa** | Botones, formularios, gráficos | Reglas de negocio, cálculos, **la base de datos** |
| **Ejemplo** | La pantalla del homebanking | Lo que verifica que tengas saldo |

**¿Dónde entra SQL?** En el back-end, siempre. Cuando alguien aprieta "Ver mis movimientos", el
front-end le pide los datos al back-end, y el back-end **hace exactamente el tipo de consulta que
venimos escribiendo en esta clase**. La consulta que escribís vos es la misma que corre adentro de
esa aplicación.

### El deploy

**Hacer un *deploy*** (o "deployar", como se dice) es **poner en producción**: pasar el código que
estaba en la computadora del programador al servidor donde lo usa la gente de verdad.

Para entenderlo hay que saber que existen **ambientes separados**:

| Ambiente | Para qué sirve | Si rompés algo acá… |
|---|---|---|
| **Desarrollo** | Donde el programador prueba | No pasa nada |
| **Testing / QA** | Donde se verifica que ande | Lo encuentra el equipo de prueba |
| **Producción** | El sistema real, con datos reales | **Lo sufren los clientes** |

> 🚨 **La regla de oro, y la razón por la que esto entra en una clase de datos:** nunca se prueba una
> consulta en producción. Un `SELECT` mal escrito puede trabar la base; un `UPDATE` o un `DELETE` sin
> `WHERE` **borra o pisa datos reales**. Si alguna vez te dan acceso a la base de una empresa,
> preguntá **siempre** en qué ambiente estás parado antes de escribir nada.

### Otras palabras que vas a escuchar

| Palabra | Qué significa |
|---|---|
| **API** | La puerta por la que dos sistemas se hablan entre sí |
| **Repositorio** (*repo*) | Donde vive el código versionado — como este repo de la materia en GitHub |
| **Query** | Una consulta a la base. Es lo que venimos escribiendo |
| **ETL** | *Extract, Transform, Load*: traer datos, limpiarlos y cargarlos. **Es literalmente lo que hacemos en esta materia** |
| **Data warehouse** | Una base pensada para analizar, separada de la que usa el sistema día a día |
| **Bug** | Un error en el código |
| **Rollback** | Volver atrás un cambio que salió mal |

> 💡 **Por qué te lo contamos:** el perfil que buscan las organizaciones no es el que solo sabe SQL,
> sino el que **entiende dónde está parado**: de dónde salen los datos, quién los genera, qué pasa si
> se rompen. Con este vocabulario podés sentarte en esa reunión y entender de qué se está hablando.

---
## 📋 Cómo se corrige (la rúbrica)

Vale la pena tenerla presente **mientras resolvés**, no cuando ya entregaste. El examen tiene
**dos partes obligatorias** y hay que sacar **4 puntos en cada una** por separado: aprobar una y
desaprobar la otra **no alcanza**.

### Parte 1 — Escrita, en papel (5 puntos)

| Criterio | Peso | Qué se mira |
|---|---|---|
| **Interpretación** | 85% | La lectura **económica** del resultado y conclusiones pertinentes |
| **Interpretación matemática** | 15% | Que entiendas el proceso matemático que hay detrás |

### Parte 2 — Práctica, en computadora (5 puntos)

| Criterio | Peso | Qué se mira |
|---|---|---|
| **Claridad de código** | 85% | Que **ejecute de punta a punta**, orden, comentarios mínimos, funciones limpias, reproducibilidad |
| **Corrección técnica** | 10% | Resultados, signos, unidades, consistencia |
| **Presentación** | 5% | Código prolijo, gráficos legibles, **ejes y leyendas etiquetados**, nombres de variables claros |

### 🎯 Lo que esto significa en la práctica

Mirá los pesos: **el 85% de la parte escrita es interpretación**, y el 85% de la práctica es que el
código sea claro y corra. Traducido:

| ❌ No alcanza con | ✅ Lo que se evalúa |
|---|---|
| Llegar al número correcto | Explicar **qué significa** ese número para la organización |
| Que el código "funcione en mi máquina" | Que corra **de punta a punta**, de arriba a abajo, sin errores |
| Un gráfico que se ve | Un gráfico **con título, ejes etiquetados y leyenda** |

> 🚨 **El error más caro y más evitable:** entregar un notebook que no corre entero. Antes de
> entregar, hacé **Entorno de ejecución → Reiniciar y ejecutar todo**. Si falla ahí, falla en la
> corrección: es el 85% de la parte práctica.

> 📎 Las pautas completas (fechas, modalidad de entrega, nombre del archivo, materiales permitidos)
> están en `CuestionesAdministrativas/Pautas/Pautas_examen.pdf`. **Leelas antes del parcial.**

---
## 🧭 Para llevarse

```sql
SELECT   columnas, FUNCION(columna) AS alias
FROM     tabla  alias
JOIN     otra_tabla alias2  ON  alias.clave = alias2.clave
WHERE    condición sobre filas
GROUP BY columnas_no_agregadas
HAVING   condición sobre grupos
ORDER BY columna DESC
LIMIT    n
```

**Ese bloque es el 90% del SQL que vas a escribir en tu vida laboral.** El orden de las cláusulas
no es negociable: si ponés el `WHERE` después del `GROUP BY`, no anda.

**Los tres errores que más se cometen:**
1. Usar `==` en vez de `=`.
2. Olvidarse el `ON` en un `JOIN` (y quedarse con miles de filas fantasma).
3. Poner en `WHERE` una condición sobre un total, que va en `HAVING`.